# Six recognizers on the 1,939 clips

This notebook decodes the 1,939 meeting clips with the six recognizers of the thesis: Canary-1B-v2, Whisper large-v3, Whisper large-v3-turbo, Whisper large-v3-turbo-German, Cohere Transcribe and Granite Speech 4.1. Apart from the working folder and one message, its code cells are the code that produced the thesis decodes. It does not cut audio and does not read the meeting recordings.

Kernel: **NeMo ASR** (`~/nemoenv`). A GPU is required.

## Before the first run

1. Build `~/nemoenv` and apply the one-time NeMo fix `patch_nemo_py310.py` as described in `../environments/README.md`.
2. Cohere Transcribe is gated on Hugging Face. Accept its conditions on its model page, then log in once in a terminal: `~/nemoenv/bin/hf auth login`.
3. The notebook works in its own folder, `recognizers/`, where JupyterLab starts its kernel. In this folder, place `goldset/segments_v9_all.json` and `goldset/goldset_gold_147.json`, and unpack the two clip packages:
   ```bash
   python -m zipfile -e clips_v10_gold.zip .
   python -m zipfile -e clips_v10_rest.zip .
   ```
   This creates `goldset/clips_v10/<segment id>.mp3`.

## Cells

Run the code cells in order. The first line of each code cell gives its number.

1. Environment check. Stops if the kernel is not `~/nemoenv`, no GPU is visible, or torch and torchaudio have different CUDA builds.
2. Clip check and helpers. Stops if the clip folder or a gold-set clip is missing. Sets the decode order: the gold-set clips first, then the rest in manifest order.
3. Canary-1B-v2.
4. The three Whisper models.
5. Cohere Transcribe, with three fixes for transformers 5.
6. Granite Speech 4.1.
7. Packs the results into `v10_asr_results.zip`.

Each model cell writes `goldset/clip_asr_v10/<code name>__v10.json`, a map from segment id to text. The code names are `canary_diar`, `whisper_large_v3`, `whisper_turbo`, `whisper_turbo_de`, `cohere` and `granite41`. A clip that raises an error is stored with empty text and the error is printed. In cells 3 and 5, one error empties the whole batch of 16 clips.

## After a crash

A model cell saves its file while it runs and skips clips that already have text, so it can be run again. After a kernel restart, run cells 1 and 2 first.

## Output

Download `v10_asr_results.zip`. It holds the result files and `goldset/segments_v9_all.json`. The Whisper models return empty text for clips longer than 30 s; `NB_whisper_long.ipynb` decodes those clips (step 2 of `README.md`).

In [ ]:
# 1. Environment check (run first)
# Install packages in a terminal, never from a notebook cell: an install inside a
# running kernel can replace torch while it is loaded.
import sys, torch
print("python  ", sys.executable)
assert "nemoenv" in sys.executable, (
    "WRONG KERNEL. Kernel -> Change Kernel -> 'NeMo ASR'. If it is missing, rebuild it with "
    "CLUSTER_SETUP.md Part A first.")

import torchaudio, nemo
from nemo.collections.asr.models import ASRModel
print("torch     ", torch.__version__)
print("torchaudio", torchaudio.__version__)
print("nemo      ", nemo.__version__)
print("CUDA      ", torch.cuda.is_available(), "|", torch.version.cuda)
assert torch.cuda.is_available(), "no GPU visible — check the kernel and the node"
assert torch.__version__.split("+")[-1] == torchaudio.__version__.split("+")[-1], \
    "torch and torchaudio built for different CUDA versions — redo CLUSTER_SETUP.md step 2"
print("\nenvironment OK")

In [ ]:
# 2. Clip check and helpers
# The clips come pre-cut by cut_clips_v9.py --all; this cell only checks that they are present.
# Do not re-cut them here: all six systems must decode the same MP3 clips the rater heard.
import os, sys, json, time, gc
from pathlib import Path
import numpy as np, torch

sys.path.insert(0, os.getcwd())

SR      = 16000
OUTDIR  = Path("goldset/clip_asr_v10"); OUTDIR.mkdir(parents=True, exist_ok=True)
CLIPDIR = Path("goldset/clips_v10")
DEVICE  = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE   = torch.bfloat16 if DEVICE == "cuda" else torch.float32

import soundfile as sf

segs = json.loads(Path("goldset/segments_v9_all.json").read_text(encoding="utf-8"))
gold = {s["segment_id"] for s in
        json.loads(Path("goldset/goldset_gold_147.json").read_text(encoding="utf-8"))}

assert CLIPDIR.exists(), (
    f"{CLIPDIR} not found. Unzip the clip package into the working directory first:\n"
    "    python -m zipfile -e clips_v10_gold.zip .  and  python -m zipfile -e clips_v10_rest.zip .")

present = {p.stem for p in CLIPDIR.glob("*.mp3")}
missing_gold = sorted(gold - present)
assert not missing_gold, (
    f"{len(missing_gold)} of the {len(gold)} rated clips are missing from {CLIPDIR}, "
    f"e.g. {missing_gold[:3]}. The upload is incomplete — re-unzip.")

print(f"{len(segs)} segments in the manifest")
print(f"{len(present)} clips uploaded   ({len(gold & present)}/{len(gold)} rated clips present)")
if len(present) < len(segs):
    print(f"  note: {len(segs) - len(present)} manifest segments have no clip; they are skipped")

# Gold-set clips first, so a session that ends early still covers the gold set;
# the other clips follow in manifest order.
ORDER = ([s["segment_id"] for s in segs if s["segment_id"] in gold and s["segment_id"] in present]
         + [s["segment_id"] for s in segs
            if s["segment_id"] not in gold and s["segment_id"] in present])
CLIPS = [CLIPDIR / f"{sid}.mp3" for sid in ORDER]
print(f"decode order: {len(gold & present)} rated first, then {len(CLIPS) - len(gold & present)}")

# Used for the probe below: falls back to librosa if libsndfile (older than 1.1) cannot read MP3.
def _read(p):
    try:
        return sf.read(str(p), dtype="float32")
    except Exception:
        import librosa
        return librosa.load(str(p), sr=SR, mono=True)

_a, _sr = _read(CLIPS[0])
print(f"probe: {CLIPS[0].name}  {len(_a)/_sr:.1f}s @ {_sr} Hz   device={DEVICE}")
assert _sr == SR, f"clips are {_sr} Hz, expected {SR}"

CLIP_SETS = {"v10": CLIPS}
print(f"\ntotal clips: {len(CLIPS)}   device={DEVICE}")


# transformers 5 renamed the from_pretrained argument torch_dtype to dtype;
# DTYPE_KW holds the keyword that the installed version expects.
import transformers
_TF5 = int(transformers.__version__.split(".")[0]) >= 5
DTYPE_KW = {"dtype": DTYPE} if _TF5 else {"torch_dtype": DTYPE}
print(f"transformers {transformers.__version__} -> using {list(DTYPE_KW)[0]}=")

def load(p):
    a, sr = sf.read(str(p), dtype="float32")
    if a.ndim > 1: a = a.mean(1)
    return a, sr

def run_model(name, fn):
    """Decode clip by clip with fn(audio, sr, path) -> str; skips clips that already have text."""
    for variant, cdir in CLIP_SETS.items():
        clips = list(cdir)
        out_path = OUTDIR / f"{name}__{variant}.json"
        done = json.loads(out_path.read_text(encoding="utf-8")) if out_path.exists() else {}
        t0 = time.time()
        for i, p in enumerate(clips, 1):
            sid = p.stem
            if sid in done and done[sid].strip():
                continue
            try:
                a, sr = load(p)
                done[sid] = (fn(a, sr, p) or "").strip()
            except Exception as e:
                done[sid] = ""
                print(f"  !! {sid}: {type(e).__name__}: {str(e)[:70]}")
            if i % 50 == 0 or i == len(clips):
                out_path.write_text(json.dumps(done, ensure_ascii=False, indent=1), encoding="utf-8")
                print(f"  [{variant}] {i}/{len(clips)}  ({round(time.time()-t0)}s)", flush=True)
        out_path.write_text(json.dumps(done, ensure_ascii=False, indent=1), encoding="utf-8")
        filled = sum(1 for v in done.values() if v.strip())
        print(f"{name} [{variant}]: {filled}/{len(clips)} transcribed -> {out_path}")

def run_batch(name, fn):
    """Decode in batches of 16 with fn(paths) -> list[str]; skips clips that already have text."""
    for variant, cdir in CLIP_SETS.items():
        clips = list(cdir)
        out_path = OUTDIR / f"{name}__{variant}.json"
        done = json.loads(out_path.read_text(encoding="utf-8")) if out_path.exists() else {}
        todo = [p for p in clips if not done.get(p.stem, "").strip()]
        print(f"  [{variant}] {len(todo)} clips to do")
        B = 16
        for i in range(0, len(todo), B):
            chunk = todo[i:i+B]
            try:
                for p, t in zip(chunk, fn([str(p) for p in chunk])):
                    done[p.stem] = (t or "").strip()
            except Exception as e:
                print(f"  !! batch @{i}: {type(e).__name__}: {str(e)[:70]}")
                for p in chunk:
                    done.setdefault(p.stem, "")
            if (i // B) % 5 == 0 or i + B >= len(todo):
                out_path.write_text(json.dumps(done, ensure_ascii=False, indent=1), encoding="utf-8")
                print(f"  [{variant}] {min(i+B, len(todo))}/{len(todo)}", flush=True)
        out_path.write_text(json.dumps(done, ensure_ascii=False, indent=1), encoding="utf-8")
        filled = sum(1 for v in done.values() if v.strip())
        print(f"{name} [{variant}]: {filled}/{len(clips)} transcribed -> {out_path}")

In [ ]:
# 3. Canary-1B-v2
# pnc must be the string "yes"; pnc=True raises an AssertionError in NeMo.
from nemo.collections.asr.models import ASRModel

print("=== canary_diar (nvidia/canary-1b-v2) ===")
canary = ASRModel.from_pretrained("nvidia/canary-1b-v2")
canary.eval()

def canary_batch(paths):
    out = canary.transcribe(paths, source_lang="de", target_lang="de", pnc="yes",
                            batch_size=len(paths))
    return [o if isinstance(o, str) else getattr(o, "text", str(o)) for o in out]

_ps = [str(p) for p in CLIP_SETS["v10"][:2]]
print("smoke test:", [s[:80] for s in canary_batch(_ps)])

run_batch("canary_diar", canary_batch)
del canary; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# 4. The three Whisper models
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline

def build_whisper(model_id):
    m = AutoModelForSpeechSeq2Seq.from_pretrained(
            model_id, low_cpu_mem_usage=True, use_safetensors=True, **DTYPE_KW).to(DEVICE)
    pr = AutoProcessor.from_pretrained(model_id)
    return pipeline("automatic-speech-recognition", model=m, tokenizer=pr.tokenizer,
                    feature_extractor=pr.feature_extractor, device=DEVICE, **DTYPE_KW)

for name, model_id in [("whisper_large_v3", "openai/whisper-large-v3"),
                       ("whisper_turbo",    "openai/whisper-large-v3-turbo"),
                       ("whisper_turbo_de", "primeline/whisper-large-v3-turbo-german")]:
    print(f"\n=== {name} ({model_id}) ===")
    pipe = build_whisper(model_id)
    fn = lambda a, sr, p: pipe({"raw": a, "sampling_rate": sr},
                               generate_kwargs={"language": "german"})["text"]
    # Smoke test on one clip before the full run.
    _p = CLIP_SETS["v10"][0]
    _a, _sr = load(_p)
    print("  smoke test:", fn(_a, _sr, _p)[:90])
    # One pass per clip: clips longer than 30 s come back empty here;
    # NB_whisper_long.ipynb decodes them again.
    run_model(name, fn)
    del pipe; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# 5. Cohere Transcribe
# Load with AutoModelForSpeechSeq2Seq and decode with model.transcribe(): AutoModel gives
# the bare encoder, and a direct generate() call fails on decoder_attention_mask=None.
from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq

# transformers 5 fix 1: the remote code declares _keys_to_ignore_on_load_* as lists, and
# transformers 5 combines them with a set (list | set raises TypeError). Convert them to sets.
import transformers.modeling_utils as _MU
if not getattr(_MU.PreTrainedModel, "_ignore_keys_setshim", False):
    _orig_adjust = _MU.PreTrainedModel._adjust_missing_and_unexpected_keys
    def _adjust_setshim(self, *a, **kw):
        for _attr in ("_keys_to_ignore_on_load_unexpected", "_keys_to_ignore_on_load_missing"):
            _v = getattr(type(self), _attr, None)
            if isinstance(_v, list):
                setattr(type(self), _attr, set(_v))
        return _orig_adjust(self, *a, **kw)
    _MU.PreTrainedModel._adjust_missing_and_unexpected_keys = _adjust_setshim
    _MU.PreTrainedModel._ignore_keys_setshim = True
    print("applied transformers-5 list/set shim")

MODEL_ID = "CohereLabs/cohere-transcribe-03-2026"
print(f"=== cohere ({MODEL_ID}) ===")
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
# Use .to(DEVICE), not device_map=: device_map goes through accelerate, which ~/nemoenv lacks.
model = AutoModelForSpeechSeq2Seq.from_pretrained(
            MODEL_ID, trust_remote_code=True, **DTYPE_KW).to(DEVICE).eval()
print("loaded class:", type(model).__name__)  # expected: CohereAsrForConditionalGeneration
assert hasattr(model, "transcribe"), "wrong class — AutoModel gives the bare encoder"

# transformers 5 fix 2: PreTrainedModel no longer inherits GenerationMixin, so super().generate()
# inside the model's generate() fails. Re-bind the instance to a subclass that adds GenerationMixin.
from transformers.generation import GenerationMixin
if not isinstance(model, GenerationMixin):
    model.__class__ = type(type(model).__name__ + "WithGen",
                           (type(model), GenerationMixin), {})
    print("re-bound to", type(model).__name__, "so super().generate() resolves")

# transformers 5 fix 3: this class gets no generation_config at init.
# Load it from the model repo, or build it from the model config.
from transformers import GenerationConfig
if getattr(model, "generation_config", None) is None:
    try:
        model.generation_config = GenerationConfig.from_pretrained(MODEL_ID)
    except Exception:
        model.generation_config = GenerationConfig.from_model_config(model.config)
    print("attached generation_config")

def cohere_batch(paths):
    return model.transcribe(processor=processor, language="de", audio_files=list(paths),
                            punctuation=True, batch_size=16)

_ps = [str(p) for p in CLIP_SETS["v10"][:2]]
_t = cohere_batch(_ps)
print("\nsmoke test:", [s[:100] for s in _t])
assert all(s.strip() for s in _t), "smoke test empty — fix before running the whole set"

run_batch("cohere", cohere_batch)
del model; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# 6. Granite Speech 4.1
from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq

MODEL_ID = "ibm-granite/granite-speech-4.1-2b"
print(f"=== granite41 ({MODEL_ID}) ===")
gproc  = AutoProcessor.from_pretrained(MODEL_ID)
gmodel = AutoModelForSpeechSeq2Seq.from_pretrained(
             MODEL_ID, **DTYPE_KW).to(DEVICE).eval()

# The prompt names German as the output language; without it the model can translate to English.
PROMPT = ("<|audio|>Transcribe the German speech into German text, with proper punctuation "
          "and capitalization. Do not translate.")
chat   = [{"role": "user", "content": PROMPT}]
prompt = gproc.tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)

def granite_fn(a, sr, p):
    wav = torch.from_numpy(a).unsqueeze(0)
    inp = gproc(prompt, wav, device=DEVICE, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        ids = gmodel.generate(**inp, max_new_tokens=200, do_sample=False, num_beams=1)
    return gproc.tokenizer.batch_decode(ids[:, inp["input_ids"].shape[-1]:],
                                        skip_special_tokens=True)[0]

run_model("granite41", granite_fn)
del gmodel; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# 7. Pack the results for download
import zipfile
out = Path("v10_asr_results.zip")
files = sorted(OUTDIR.glob("*.json"))
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for f in files:
        z.write(f, f"goldset/clip_asr_v10/{f.name}")
    z.write("goldset/segments_v9_all.json", "goldset/segments_v9_all.json")
print(f"{out}  ({out.stat().st_size/1e6:.1f} MB)")
for f in files:
    d = json.loads(f.read_text(encoding="utf-8"))
    print(f"  {f.name:<34} {sum(1 for v in d.values() if v.strip())}/{len(d)} non-empty")
print("\nExpect 6 files, one per system. Download v10_asr_results.zip.")